# 01 · Exploración

Revisa qué hay en los zips de `dataset/`: integridad de los ficheros, metadatos, colores de máscara, geometría de las instancias, parásitos por imagen, imágenes vacías y calidad.

Kernel: **AiScope (.venv)** (ver README). El índice se construye leyendo los zips sin descomprimir (`aiscope.data.index`) y se guarda en `data/interim/`. La primera ejecución recorre todas las máscaras a resolución completa; las siguientes leen la caché.

La leyenda sale de la app de etiquetado [GDD-app](https://github.com/theaiscope/GDD-app) y está en `aiscope.data.classes`: color → estadio (anillo, trofozoíto, esquizonte, gametocito, artefacto), `species` → especie de *Plasmodium* y `bloodType` → preparación (fina o gruesa). **El artefacto no es un parásito**: no cuenta en el conteo.

In [ ]:
import sys
try:
    import aiscope  # noqa: F401
except ModuleNotFoundError:
    raise RuntimeError(f"Kernel equivocado ({sys.executable}). Usa «AiScope (.venv)»: Kernel → Change kernel.") from None

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from PIL import ImageDraw

from aiscope.paths import RAW_DIR, INTERIM_DIR
from aiscope.data.index import build_index, to_image_coords
from aiscope.data.viz import instance_crop, thumbnail, grid
from aiscope.data.classes import MASK_COLORS, STAGE_ES, SPECIES, SPECIES_SHORT, SMEAR, NOT_PARASITE
from aiscope import style

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
style.apply()

samples, images, instances = build_index(RAW_DIR, INTERIM_DIR)
samples["year"] = samples["created_on"].str[:4]
samples["created_day"] = samples["created_on"].str[:10]
samples["species_code"] = samples["species"]
samples["species"] = samples["species"].map(SPECIES)
samples["blood_type"] = samples["blood_type"].map(SMEAR)
meta_cols = ["folder", "species", "species_code", "blood_type", "health_facility", "device", "microscopist", "year"]
images = images.merge(samples[meta_cols], on="folder", how="left")
instances = to_image_coords(instances, images)  # hay máscaras guardadas a media resolución
instances = instances.merge(images[["image_id", "img_w", "img_h", "species", "species_code", "blood_type", "health_facility"]], on="image_id", how="left")
instances["estadio"] = instances["color"].map(lambda c: STAGE_ES.get(MASK_COLORS.get(c), f"desconocido {c}"))
instances["es_parasito"] = ~instances["color"].map(MASK_COLORS).isin(NOT_PARASITE)
print(f"{len(samples)} carpetas · {len(images)} filas imagen/máscara · {len(instances)} instancias")

## 1. Integridad de ficheros

In [ ]:
pair = images["image_file"].notna() & images["mask_file"].notna()
read_err = images["read_error"].notna() if "read_error" in images else pd.Series(False, index=images.index)
size_mismatch = pair & ((images["mask_w"] != images["img_w"]) | (images["mask_h"] != images["img_h"]))
integridad = pd.Series({
    "zips": samples["zip"].nunique(),
    "carpetas": len(samples),
    "carpetas vacías": int(((samples["n_image_files"] == 0) & (samples["n_mask_files"] == 0)).sum()),
    "carpetas sin metadata.json": int((~samples["has_metadata"]).sum()),
    "carpetas con ficheros inesperados": int(samples["other_files"].map(len).gt(0).sum()),
    "imágenes": int(images["image_file"].notna().sum()),
    "máscaras": int(images["mask_file"].notna().sum()),
    "pares imagen + máscara": int(pair.sum()),
    "imágenes sin máscara": int((images["image_file"].notna() & images["mask_file"].isna()).sum()),
    "máscaras sin imagen": int((images["image_file"].isna() & images["mask_file"].notna()).sum()),
    "errores de lectura": int(read_err.sum()),
    "máscara de tamaño distinto a la imagen": int(size_mismatch.sum()),
    "EXIF orientation ≠ 1": int((images["exif_orientation"].fillna(1) != 1).sum()),
})
display(integridad.to_frame("valor"))
display(images.loc[~pair | read_err | size_mismatch, ["zip", "folder", "idx", "image_file", "mask_file"]])

## 2. Metadatos por carpeta

`blood_type` es el tipo de preparación (fina o gruesa) y `species` la especie de *Plasmodium*. `sample_age = old` indica una muestra antigua del laboratorio, no fresca. Una carpeta no corresponde a un paciente.

In [ ]:
samples["n_imagenes"] = samples["folder"].map(images[pair].groupby("folder").size()).fillna(0).astype(int)
display(samples["n_imagenes"].describe().to_frame("imágenes por carpeta").T)
fig, ax = plt.subplots(figsize=(8, 3))
samples["n_imagenes"].plot.hist(bins=range(0, samples["n_imagenes"].max() + 2), ax=ax)
ax.set_xlabel("imágenes por carpeta"); ax.set_ylabel("carpetas")
plt.show()

def resumen(col):
    return (samples.groupby(col, dropna=False)
            .agg(carpetas=("folder", "size"), imagenes=("n_imagenes", "sum"))
            .sort_values("carpetas", ascending=False))

for col in ["species", "blood_type", "sample_age", "year", "magnification"]:
    display(resumen(col))

In [ ]:
display(pd.crosstab(samples["species"], samples["blood_type"], values=samples["n_imagenes"], aggfunc="sum", margins=True).fillna(0).astype(int)
        .rename_axis(index="species \\ blood_type (imágenes)"))
display(pd.crosstab(samples["species"], samples["sample_age"].fillna("sin dato"), values=samples["n_imagenes"], aggfunc="sum", margins=True).fillna(0).astype(int)
        .rename_axis(index="species \\ sample_age (imágenes)"))
display(samples.loc[samples["comments"].str.len() > 0, ["folder", "species", "blood_type", "comments"]])

Agrupación contra la fuga entre splits. Sin ID de paciente ni de lámina, se toma como grupo la sesión de captura: mismo centro, microscopista y día. Aquí solo se mide su tamaño.

In [ ]:
ses = samples.groupby(["health_facility", "microscopist", "created_day"]).agg(
    carpetas=("folder", "size"), imagenes=("n_imagenes", "sum"),
    especies=("species", "nunique"), tipos=("blood_type", "nunique"))
display(ses["carpetas"].describe(percentiles=[.5, .9, .99]).to_frame("carpetas por sesión").T)
print("sesiones:", len(ses), "· sesiones con >1 carpeta:", int((ses["carpetas"] > 1).sum()),
      "· carpetas en sesiones con >1 carpeta:", int(ses.loc[ses["carpetas"] > 1, "carpetas"].sum()))

## 3. Colores de máscara y estadios

Se cuentan como instancia las componentes conexas (8-conectividad) de un color RGB exacto con alpha = 255 y al menos 30 px.

In [ ]:
paleta = (instances.groupby("color")
          .agg(instancias=("inst", "size"), imagenes=("image_id", "nunique"), carpetas=("folder", "nunique"),
               area_mediana=("area", "median"))
          .sort_values("instancias", ascending=False))
paleta["% instancias"] = (100 * paleta["instancias"] / paleta["instancias"].sum()).round(2)
paleta.insert(0, "estadio", [STAGE_ES.get(MASK_COLORS.get(c), "desconocido") for c in paleta.index])
display(paleta)

fig, ax = plt.subplots(figsize=(7, 0.3 * len(paleta) + 1))
ax.barh(paleta["estadio"][::-1], paleta["instancias"][::-1], color=list(paleta.index[::-1]), edgecolor="k")
ax.set_xscale("log"); ax.set_xlabel("instancias (log)")
plt.show()

def rgb(h):
    return np.array([int(h[i:i + 2], 16) for i in (1, 3, 5)])

principales = paleta.index[paleta["instancias"] >= 20].tolist()
menores = [c for c in paleta.index if c not in principales]
print("colores principales (≥20 instancias):", principales)
print("colores fuera de la leyenda de GDD-app:", [c for c in paleta.index if c not in MASK_COLORS])
if menores:
    display(pd.DataFrame([{
        "color": c,
        "principal más cercano": min(principales, key=lambda p: np.abs(rgb(c) - rgb(p)).sum()),
        "distancia L1": min(int(np.abs(rgb(c) - rgb(p)).sum()) for p in principales),
        "instancias": int(paleta.loc[c, "instancias"]),
    } for c in menores]).sort_values("instancias", ascending=False))
display(images.loc[pair, ["opaque_px", "stray_px", "partial_alpha_px", "n_colors"]].describe(percentiles=[.5, .95, .99]).T)

Recortes al azar de cada color principal, con el contorno de la máscara en rojo. Encima de cada recorte figuran la especie y la preparación de su carpeta.

In [ ]:
img_idx = images.set_index("image_id", drop=False)

def rotulado(img_row, inst_row, txt):
    im = instance_crop(RAW_DIR, img_row, inst_row)
    d = ImageDraw.Draw(im)
    d.rectangle([0, 0, 90, 14], fill=(255, 255, 255))
    d.text((2, 1), txt, fill=(0, 0, 0))
    return im

for c in principales:
    sub = instances[instances["color"] == c]
    pick = sub.sample(min(12, len(sub)), random_state=0)
    crops = [rotulado(img_idx.loc[r["image_id"]], r, f"{SPECIES_SHORT[r['species_code']]} {r['blood_type']}") for _, r in pick.iterrows()]
    fig, ax = plt.subplots(figsize=(14, 5.2))
    ax.imshow(grid(crops, 6)); ax.axis("off")
    ax.set_title(f"{STAGE_ES.get(MASK_COLORS.get(c), c)} ({c}) · {len(sub)} instancias", color=c, fontsize=13, loc="left")
    plt.show()

## 4. Distribución de instancias por estadio, especie y preparación

In [ ]:
inst_p = instances[instances["color"].isin(principales)].copy()
print(f"parásitos: {int(inst_p['es_parasito'].sum())} · artefactos: {int((~inst_p['es_parasito']).sum())}")
display(pd.crosstab(inst_p["estadio"], inst_p["species"], margins=True).rename_axis(index="estadio \\ especie"))
display(pd.crosstab(inst_p["estadio"], inst_p["blood_type"], margins=True).rename_axis(index="estadio \\ preparación"))
display(pd.crosstab([inst_p["blood_type"], inst_p["species"]], inst_p["estadio"], margins=True))
display(inst_p.groupby(["species", "estadio"])["folder"].nunique().unstack(fill_value=0)
        .rename_axis(index="carpetas con instancias: especie \\ estadio"))

conteo = inst_p["estadio"].value_counts()
comb = inst_p[inst_p["es_parasito"]].groupby(["blood_type", "species", "estadio"]).size()
print(f"desequilibrio entre estadios (max/min): {conteo.max() / conteo.min():.0f}x")
print(f"desequilibrio preparación×especie×estadio de parásitos (max/min, combinaciones presentes): {comb.max() / comb.min():.0f}x")
conc = (inst_p.groupby(["estadio", "folder"]).size()
        .groupby(level=0).apply(lambda s: s.nlargest(5).sum() / s.sum()))
display((100 * conc).round(1).rename("% de instancias en sus 5 carpetas con más").to_frame())

## 5. Geometría de las instancias

`lado` es el lado mayor de la caja que envuelve el trazo anotado. En GDD-app el pincel mide `80 px / zoom` (`MaskLayer.kt`): un toque deja un disco cuyo tamaño depende del zoom del anotador, no del parásito. Por eso la caja del trazo es una aproximación holgada y de tamaño variable.

In [ ]:
g = inst_p.merge(images[["image_id", "field_x0", "field_y0", "field_x1", "field_y1"]], on="image_id", how="left")
g["w"] = g["x1"] - g["x0"]
g["h"] = g["y1"] - g["y0"]
g["lado"] = g[["w", "h"]].max(axis=1)
g["aspecto"] = g["lado"] / g[["w", "h"]].min(axis=1)
g["relleno"] = g["area"] / (g["w"] * g["h"])
g["lado_campo"] = np.maximum(g["field_x1"] - g["field_x0"], g["field_y1"] - g["field_y0"])

display(g.groupby(["blood_type", "estadio"])["lado"].describe(percentiles=[.05, .5, .95]).round(0))
display(g.groupby("estadio")[["relleno", "aspecto"]].describe(percentiles=[.05, .5, .95]).round(2))
print(f"instancias con hueco interior: {(g['hole_px'] > 0).mean():.1%}  (relleno de una elipse llena ≈ 0.785)")

Tamaño aparente de las instancias según cómo se prepare la entrada del modelo.

In [ ]:
escalas = {
    "original": 1.0,
    "imagen a 1/2": 0.5,
    "imagen a 1/4": 0.25,
    "imagen entera a 640": 640 / g[["img_w", "img_h"]].max(axis=1),
    "campo recortado a 640": 640 / g["lado_campo"],
    "campo recortado a 1280": 1280 / g["lado_campo"],
}
tam = pd.DataFrame({k: g["lado"] * v for k, v in escalas.items()})
for bt, sub in tam.groupby(g["blood_type"]):
    res = pd.DataFrame({
        "mediana px": sub.median(), "p5 px": sub.quantile(.05),
        "% < 12 px": 100 * (sub < 12).mean(), "% < 24 px": 100 * (sub < 24).mean(),
    }).round(1)
    display(res.rename_axis(index=f"preparación {bt}"))
display(images.loc[pair, ["field_frac"]].assign(
    lado_campo=np.maximum(images["field_x1"] - images["field_x0"], images["field_y1"] - images["field_y0"])).describe(percentiles=[.05, .5, .95]).T)

Tres anomalías de anotación que se tratan en `02_limpieza`:
- **Área atípica**: el trazo mide más de 2.5 veces la mediana de su estadio y preparación. Con poco zoom el pincel es grueso y la caja queda muy holgada.
- **Fusión**: dos trazos del mismo color se tocan y forman una sola componente. Se detecta por un relleno bajo (< 0.65) en colores cuyo trazo es normalmente elíptico.
- **Fragmento**: área por debajo de 0.1 veces la mediana.

In [ ]:
g["area_rel"] = g["area"] / g.groupby(["blood_type", "color"])["area"].transform("median")
atipicas = g[g["area_rel"] > 2.5]
eliptico = g.groupby("color")["relleno"].transform("median") > 0.74
fusion = g[eliptico & (g["relleno"] < 0.65)]
fragm = g[g["area_rel"] < 0.1]
print(f"área atípica (> 2.5× mediana): {len(atipicas)} ({len(atipicas) / len(g):.2%})")
display(atipicas.groupby(["blood_type", "estadio"]).size().unstack(fill_value=0).rename_axis(index="área atípica: preparación \\ estadio"))
print(f"posibles fusiones (relleno < 0.65 en colores elípticos): {len(fusion)} ({len(fusion) / len(g):.2%})")
print(f"posibles fragmentos (área < 0.1× mediana): {len(fragm)} ({len(fragm) / len(g):.2%})")
for titulo, df in [("área atípica (mayor ratio)", atipicas.nlargest(12, "area_rel")),
                   ("área atípica (ratio 2.5–4, al azar)", atipicas[atipicas["area_rel"] < 4].sample(min(12, len(atipicas)), random_state=0)),
                   ("posibles fusiones (menor relleno)", fusion.nsmallest(12, "relleno")),
                   ("fragmentos", fragm.nsmallest(12, "area_rel"))]:
    if len(df):
        crops = [rotulado(img_idx.loc[r["image_id"]], r, f"x{r['area_rel']:.2f}") for _, r in df.iterrows()]
        fig, ax = plt.subplots(figsize=(14, 5.2)); ax.imshow(grid(crops, 6)); ax.axis("off"); ax.set_title(titulo, loc="left"); plt.show()

In [ ]:
def solapes(df):
    b = df[["x0", "y0", "x1", "y1"]].to_numpy(float)
    iw = np.clip(np.minimum(b[:, None, 2], b[None, :, 2]) - np.maximum(b[:, None, 0], b[None, :, 0]), 0, None)
    ih = np.clip(np.minimum(b[:, None, 3], b[None, :, 3]) - np.maximum(b[:, None, 1], b[None, :, 1]), 0, None)
    inter = iw * ih
    area = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    i, j = np.triu_indices(len(b), 1)
    iou = inter[i, j] / (area[i] + area[j] - inter[i, j])
    contenida = inter[i, j] / np.minimum(area[i], area[j])
    col = df["color"].to_numpy()
    return pd.DataFrame({"iou": iou, "contenida": contenida, "mismo_color": col[i] == col[j]})

multi = g.groupby("image_id").filter(lambda d: len(d) > 1)
pares_inst = pd.concat([solapes(d) for _, d in multi.groupby("image_id")], ignore_index=True)
display(pd.DataFrame({
    "IoU > 0.3": pares_inst[pares_inst["iou"] > 0.3].groupby("mismo_color").size(),
    "caja contenida > 90%": pares_inst[pares_inst["contenida"] > 0.9].groupby("mismo_color").size(),
}).fillna(0).astype(int).rename_axis(index="mismo color"))

## 6. Parásitos por imagen e imágenes sin parásitos

Solo cuentan los estadios de parásito; los artefactos van aparte.

In [ ]:
pares = images[pair].copy()
pares["n_inst"] = pares["image_id"].map(inst_p[inst_p["es_parasito"]].groupby("image_id").size()).fillna(0).astype(int)
pares["n_artefactos"] = pares["image_id"].map(inst_p[~inst_p["es_parasito"]].groupby("image_id").size()).fillna(0).astype(int)
display(pares["n_inst"].describe(percentiles=[.5, .9, .95, .99]).to_frame("parásitos por imagen").T)
fig, ax = plt.subplots(figsize=(9, 3))
pares["n_inst"].clip(upper=40).value_counts().sort_index().plot.bar(ax=ax)
ax.set_xlabel("parásitos por imagen (40 = ≥40)"); ax.set_ylabel("imágenes")
plt.show()

vacias = pares["n_inst"] == 0
print(f"imágenes sin parásitos: {int(vacias.sum())} ({vacias.mean():.1%}) · de ellas con algún artefacto: {int((vacias & (pares['n_artefactos'] > 0)).sum())}")
display(pares.groupby(["blood_type", "species"])["n_inst"].agg(
    imagenes="size", media="mean", mediana="median", p95=lambda s: s.quantile(.95), max="max", vacias=lambda s: int((s == 0).sum())).round(1))
por_carpeta = pares.groupby("folder")["n_inst"].agg(["size", "sum"])
print("carpetas con todas sus imágenes vacías:", int((por_carpeta["sum"] == 0).sum()))
cnt_estadio = inst_p[inst_p["es_parasito"]].groupby(["image_id", "estadio"]).size().unstack(fill_value=0)
print("imágenes con más de un estadio de parásito:", int((cnt_estadio > 0).sum(axis=1).gt(1).sum()))
if vacias.any():
    muestra = pares[vacias].sample(min(8, int(vacias.sum())), random_state=0)
    fig, ax = plt.subplots(figsize=(14, 5)); ax.imshow(grid([thumbnail(RAW_DIR, r, 256) for _, r in muestra.iterrows()], 8)); ax.axis("off")
    ax.set_title("imágenes sin parásitos (muestra)", loc="left"); plt.show()

## 7. Calidad de imagen

In [ ]:
pares["resolucion"] = pares["img_w"].astype(int).astype(str) + "×" + pares["img_h"].astype(int).astype(str)
display(pares["resolucion"].value_counts().rename("imágenes").to_frame())

fig, axes = plt.subplots(1, 4, figsize=(16, 3))
pares["field_frac"].plot.hist(bins=50, ax=axes[0]); axes[0].set_xlabel("fracción de imagen dentro del campo")
np.log10(pares["sharpness"]).plot.hist(bins=50, ax=axes[1]); axes[1].set_xlabel("log10 nitidez (var. laplaciano)")
pares["brightness"].plot.hist(bins=50, ax=axes[2]); axes[2].set_xlabel("brillo medio del campo")
for bt, sub in pares.groupby("blood_type"):
    axes[3].scatter(sub["mean_r"] - sub["mean_b"], sub["brightness"], s=3, label=f"gota {bt}", alpha=.5)
axes[3].set_xlabel("R − B medio (tinción)"); axes[3].set_ylabel("brillo"); axes[3].legend(fontsize=6, markerscale=3)
plt.tight_layout(); plt.show()
display(pares.groupby("blood_type")[["sharpness", "brightness", "contrast", "field_frac"]].median().round(2))

In [ ]:
for titulo, df in [("menos nítidas", pares.nsmallest(8, "sharpness")),
                   ("más oscuras", pares.nsmallest(8, "brightness")),
                   ("menor fracción de campo", pares.nsmallest(8, "field_frac"))]:
    fig, ax = plt.subplots(figsize=(14, 3.2))
    ax.imshow(grid([thumbnail(RAW_DIR, r, 256) for _, r in df.iterrows()], 8)); ax.axis("off"); ax.set_title(titulo, loc="left")
    plt.show()

Casi duplicados: dHash de 256 bits sobre el centro del campo. Los que caen en carpetas distintas son riesgo de fuga entre splits.

In [ ]:
H = np.stack([np.frombuffer(bytes.fromhex(h), dtype=np.uint8) for h in pares["dhash"]])
B = np.unpackbits(H, axis=1).astype(np.float32)
pc = B.sum(1)
D = pc[:, None] + pc[None, :] - 2 * (B @ B.T)
np.fill_diagonal(D, 1e9)
fold = pares["folder"].to_numpy()
same = fold[:, None] == fold[None, :]
nn_otro = np.where(same, 1e9, D).min(1)
nn_misma = np.where(same, D, 1e9).min(1)

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(nn_misma[nn_misma < 1e9], bins=range(0, 130, 2), alpha=.6, label="vecino más cercano en la misma carpeta")
ax.hist(nn_otro, bins=range(0, 130, 2), alpha=.6, label="vecino más cercano en otra carpeta")
ax.set_xlabel("distancia de Hamming (de 256)"); ax.legend(); plt.show()

UMBRAL = 12
i, j = np.where(np.triu(D <= UMBRAL, 1))
dup = pd.DataFrame({"a": pares["image_id"].to_numpy()[i], "b": pares["image_id"].to_numpy()[j], "d": D[i, j], "misma_carpeta": same[i, j]})
sesion = (samples["health_facility"] + "|" + samples["microscopist"] + "|" + samples["created_day"]).set_axis(samples["folder"])
dup["misma_sesion"] = dup["a"].str.split("/").str[0].map(sesion).to_numpy() == dup["b"].str.split("/").str[0].map(sesion).to_numpy()
print(f"pares con Hamming ≤ {UMBRAL}: {len(dup)} · en la misma carpeta: {int(dup['misma_carpeta'].sum())} · entre carpetas: {int((~dup['misma_carpeta']).sum())}"
      f" · entre sesiones distintas: {int((~dup['misma_sesion']).sum())}")
cruz = dup[~dup["misma_carpeta"]].nsmallest(6, "d")
if len(cruz):
    ims = []
    for _, r in cruz.iterrows():
        ims += [thumbnail(RAW_DIR, img_idx.loc[r["a"]], 256), thumbnail(RAW_DIR, img_idx.loc[r["b"]], 256)]
    fig, ax = plt.subplots(figsize=(14, 2.6 * len(cruz))); ax.imshow(grid(ims, 4)); ax.axis("off")
    ax.set_title("pares entre carpetas más parecidos (izquierda / derecha)", loc="left"); plt.show()

## 8. Resumen numérico

In [ ]:
resumen = pd.Series({
    "carpetas": len(samples),
    "pares imagen + máscara": int(pair.sum()),
    "parásitos anotados": int(inst_p["es_parasito"].sum()),
    "artefactos anotados": int((~inst_p["es_parasito"]).sum()),
    "imágenes sin parásitos": int(vacias.sum()),
    "mediana parásitos / imagen": float(pares["n_inst"].median()),
    "p95 parásitos / imagen": float(pares["n_inst"].quantile(.95)),
    "estadio más frecuente / menos frecuente": f"{conteo.max() / conteo.min():.0f}x",
    "grupos anti-fuga (sesiones)": len(ses),
    "pares casi duplicados entre carpetas": int((~dup["misma_carpeta"]).sum()),
    "pares casi duplicados entre sesiones": int((~dup["misma_sesion"]).sum()),
    "instancias de área atípica": len(atipicas),
    "posibles fusiones": len(fusion),
    "máscaras a otra resolución": int(size_mismatch.sum()),
})
display(resumen.to_frame("valor"))